# 06 - Avaliação Final e Conclusões
## Relatório completo de performance, análise de risco e limitações
Este notebook consolida todos os resultados do projeto e gera
um relatório detalhado comparando a estratégia ML com o benchmark.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import quantstats as qs
import warnings
warnings.filterwarnings("ignore")

portfolio = pd.read_parquet("../data/processed/portfolio_returns.parquet")
predictions = pd.read_parquet("../data/processed/predictions.parquet")

print(f"Retornos do portfólio: {portfolio.shape}")
print(f"Previsões: {predictions.shape}")
print(f"Período: {portfolio.index.min()} até {portfolio.index.max()}")

### 6.1 Preparar séries de retorno
O quantstats espera séries com DatetimeIndex e sem valores nulos.

In [ ]:
strategy_returns = portfolio["portfolio_return"].copy()
strategy_returns.index = pd.to_datetime(strategy_returns.index)
strategy_returns = strategy_returns.dropna()
strategy_returns.index.name = "Date"
strategy_returns.name = "ML Strategy"

print(f"Dias de retorno: {len(strategy_returns)}")
print(f"Retorno médio diário: {strategy_returns.mean():.6f}")
print(f"Desvio padrão diário: {strategy_returns.std():.6f}")

### 6.2 Relatório quantstats em HTML
Geramos um relatório interativo completo comparando com o SPY.
O arquivo HTML será salvo na pasta results.

In [ ]:
qs.reports.html(
    strategy_returns,
    benchmark="SPY",
    output="../results/quantstats_report.html",
    title="ML Trading Strategy vs SPY"
)
print("Relatório salvo em results/quantstats_report.html")

### 6.3 Métricas resumidas via quantstats

In [ ]:
print("=== Métricas da Estratégia ML ===\n")
qs.reports.metrics(strategy_returns, benchmark="SPY", mode="full")

### 6.4 Análise de retornos por regime de mercado
Avaliamos como a estratégia performa em cada regime identificado
no notebook 03 (alta, baixa, crise).

In [ ]:
features = pd.read_parquet("../data/processed/features_with_regime.parquet")

regime_daily = features.groupby(level="date")["market_regime_name"].first()
regime_daily.index = pd.to_datetime(regime_daily.index)

common = strategy_returns.index.intersection(regime_daily.index)
regime_aligned = regime_daily.loc[common]
returns_aligned = strategy_returns.loc[common]

regime_perf = pd.DataFrame({
    "retorno_diario_medio": returns_aligned.groupby(regime_aligned).mean(),
    "retorno_diario_std": returns_aligned.groupby(regime_aligned).std(),
    "n_dias": returns_aligned.groupby(regime_aligned).count(),
    "retorno_anualizado": returns_aligned.groupby(regime_aligned).mean() * 252,
    "vol_anualizada": returns_aligned.groupby(regime_aligned).std() * np.sqrt(252),
})

regime_perf["sharpe"] = (regime_perf["retorno_anualizado"] - 0.04) / regime_perf["vol_anualizada"]

print("Performance por regime de mercado:")
print(regime_perf.round(4).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

colors = {"alta": "green", "baixa": "orange", "crise": "red"}

for ax, (regime, group) in zip(axes, returns_aligned.groupby(regime_aligned)):
    ax.hist(group, bins=50, color=colors.get(regime, "gray"), alpha=0.7, edgecolor="white")
    ax.axvline(x=group.mean(), color="black", linestyle="--", linewidth=1)
    ax.set_title(f"Regime: {regime} (n={len(group)})")
    ax.set_xlabel("Retorno diário")

plt.suptitle("Distribuição dos retornos por regime de mercado", fontsize=13)
plt.tight_layout()
plt.savefig("../results/returns_by_regime.png", dpi=150, bbox_inches="tight")
plt.show()

### 6.5 Análise de consistência temporal
Retorno anual da estratégia vs SPY, ano a ano.

In [ ]:
annual_strategy = strategy_returns.resample("YE").apply(lambda x: (1 + x).prod() - 1)
annual_strategy.index = annual_strategy.index.year

import yfinance as yf
spy = yf.download("SPY",
    start=strategy_returns.index.min().strftime("%Y-%m-%d"),
    end=(strategy_returns.index.max() + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
    auto_adjust=False
)
if isinstance(spy.columns, pd.MultiIndex):
    spy.columns = spy.columns.droplevel(1)
spy_ret = spy["Close"].pct_change().dropna()
spy_ret.index = pd.to_datetime(spy_ret.index)
spy_annual = spy_ret.resample("YE").apply(lambda x: (1 + x).prod() - 1)
spy_annual.index = spy_annual.index.year

common_years = annual_strategy.index.intersection(spy_annual.index)
annual_strategy = annual_strategy.loc[common_years]
spy_annual = spy_annual.loc[common_years]

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(common_years))
width = 0.35

bars1 = ax.bar(x - width/2, annual_strategy.values, width, label="ML Strategy", color="steelblue")
bars2 = ax.bar(x + width/2, spy_annual.values, width, label="SPY", color="gray")

ax.set_xlabel("Ano")
ax.set_ylabel("Retorno anual")
ax.set_title("Retorno anual: ML Strategy vs SPY")
ax.set_xticks(x)
ax.set_xticklabels(common_years, rotation=45)
ax.legend()
ax.axhline(y=0, color="black", linewidth=0.5)
ax.grid(axis="y", alpha=0.3)

for bar in bars1:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h, f"{h:.0%}",
            ha="center", va="bottom" if h >= 0 else "top", fontsize=8)

plt.tight_layout()
plt.savefig("../results/annual_returns_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

wins = (annual_strategy > spy_annual).sum()
total = len(common_years)
print(f"\nAnos em que a estratégia bateu o SPY: {wins}/{total} ({wins/total:.0%})")

### 6.6 Análise de risco: piores períodos

In [ ]:
cumulative = (1 + strategy_returns).cumprod()
running_max = cumulative.cummax()
drawdown = (cumulative - running_max) / running_max

worst_dd_end = drawdown.idxmin()
dd_start_candidates = cumulative.loc[:worst_dd_end]
worst_dd_start = dd_start_candidates.idxmax()
worst_dd_value = drawdown.min()
recovery_candidates = cumulative.loc[worst_dd_end:]
recovery_mask = recovery_candidates >= cumulative.loc[worst_dd_start]
if recovery_mask.any():
    recovery_date = recovery_mask.idxmax()
    recovery_days = (recovery_date - worst_dd_end).days
else:
    recovery_date = "Não recuperou"
    recovery_days = "N/A"

print("=== Pior drawdown ===")
print(f"Início: {worst_dd_start.date()}")
print(f"Fundo: {worst_dd_end.date()}")
print(f"Magnitude: {worst_dd_value:.2%}")
print(f"Recuperação: {recovery_date}")
if isinstance(recovery_days, int):
    print(f"Dias para recuperar: {recovery_days}")

print("\n=== 5 piores dias ===")
worst_days = strategy_returns.nsmallest(5)
for date, ret in worst_days.items():
    print(f"  {date.date()}: {ret:.2%}")

print("\n=== 5 melhores dias ===")
best_days = strategy_returns.nlargest(5)
for date, ret in best_days.items():
    print(f"  {date.date()}: {ret:.2%}")

### 6.7 Resumo das métricas finais

In [ ]:
def calc_metrics(returns, name):
    total_return = (1 + returns).prod() - 1
    n_years = len(returns) / 252
    annual_return = (1 + total_return) ** (1 / n_years) - 1
    annual_vol = returns.std() * np.sqrt(252)
    sharpe = (annual_return - 0.04) / annual_vol
    downside = returns[returns < 0].std() * np.sqrt(252)
    sortino = (annual_return - 0.04) / downside
    cum = (1 + returns).cumprod()
    dd = (cum - cum.cummax()) / cum.cummax()
    max_dd = dd.min()
    calmar = annual_return / abs(max_dd) if max_dd != 0 else np.inf
    win_rate = (returns > 0).mean()
    return {
        "Retorno total": f"{total_return:.2%}",
        "Retorno anual": f"{annual_return:.2%}",
        "Volatilidade anual": f"{annual_vol:.2%}",
        "Sharpe Ratio": f"{sharpe:.2f}",
        "Sortino Ratio": f"{sortino:.2f}",
        "Max Drawdown": f"{max_dd:.2%}",
        "Calmar Ratio": f"{calmar:.2f}",
        "Win Rate (dias)": f"{win_rate:.1%}"
    }

spy_common = spy_ret.loc[strategy_returns.index.intersection(spy_ret.index)]

m_strat = calc_metrics(strategy_returns, "ML Strategy")
m_spy = calc_metrics(spy_common, "SPY")

summary = pd.DataFrame({"ML Strategy": m_strat, "SPY (Buy & Hold)": m_spy})
print(summary.to_string())

summary.to_csv("../results/final_metrics.csv")
print("\nSalvo em results/final_metrics.csv")

### 6.8 Limitações e considerações

**Survivorship bias**: usamos a composição atual do S&P 500 para todo o período
desde 2010. Empresas que faliram ou foram removidas do índice não estão na amostra,
o que pode inflacionar artificialmente os resultados.

**Custos de transação simplificados**: aplicamos um custo fixo de 0.1% sobre o
turnover, mas na prática os custos variam com o tamanho da ordem, liquidez da ação
e condições de mercado. Slippage e impacto de mercado não foram modelados.

**Turnover alto**: a estratégia troca grande parte do portfólio a cada mês.
Isso gera custos reais e impostos sobre ganhos de capital que não foram considerados.

**Sem dados alternativos**: o modelo usa apenas dados de preço, volume e fatores
Fama-French. Dados fundamentais (lucros, receita), de sentimento ou macroeconômicos
poderiam melhorar as previsões.

**Look-ahead bias no filtro de liquidez**: o filtro de 150 ações mais líquidas
usa a média de dollar volume do mês corrente, o que inclui informação do futuro.
Uma implementação mais rigorosa usaria o mês anterior.

**Regime de mercado predominantemente de alta**: o período testado (2013-2026)
coincide com um bull market prolongado nos EUA. A estratégia não foi testada em
períodos de bear market sustentado (ex: 2000-2002, 2007-2009).

**Overfitting potencial**: apesar do walk-forward validation, o XGBoost com 37
features pode capturar padrões espúrios. Uma validação mais robusta incluiria
análise de sensibilidade dos hiperparâmetros e testes em outros mercados.

### 6.9 Conclusão

Este projeto implementou uma estratégia quantitativa de trading usando
aprendizado de máquina, cobrindo o pipeline completo: coleta de dados,
engenharia de features, detecção de regimes de mercado via clusterização,
modelagem preditiva com XGBoost, otimização de portfólio via Efficient Frontier,
e avaliação com métricas de risco adequadas.

Os resultados mostram que o modelo possui capacidade preditiva (correlação de
Spearman positiva em 65% dos dias, spread monotônico entre quintis). O portfólio
construído superou o benchmark SPY em retorno ajustado ao risco, com Sharpe Ratio
superior e drawdown máximo comparável.

No entanto, os resultados devem ser interpretados com cautela dadas as limitações
mencionadas acima. Este é um projeto de portfólio pessoal e estudo, não uma
recomendação de investimento.